In [1]:
from langchain.tools import tool

@tool
def add_numbers(a: float, b: float) -> float:
    """Calculates the sum of two numbers."""
    return a+b

@tool
def Substract_numbers(a: float, b: float) -> float:
    """Substracts one number from another number."""
    return a-b

@tool
def multiply(a: float, b: float) -> float:
    """Multiplies two numbers."""
    return a*b

In [2]:
tools = [Substract_numbers, add_numbers,multiply]

In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(tools)


In [4]:
#building workflow in LangGraph
from typing_extensions import TypedDict #for entire State
from langchain_core.messages import AnyMessage #human or AI message
from typing import Annotated  #labeling
from langgraph.graph.message import add_messages #add messages is reducer in langgraph
from IPython.display import display,Image
from langgraph.graph import StateGraph,START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

In [5]:
# 1. Define the State clearly
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [6]:
from langchain_core.messages.utils import convert_to_openai_messages
def tool_calling_llm(state: State):
    # Automatically handles roles, tool calls, and content blocks
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
# 2. Build the graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")

# 3. Use the explicit mapping in conditional edges
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
    {
        "tools": "tools",
        "__end__": END
    }
)

# 4. IMPORTANT: Loop back to LLM after tool execution
builder.add_edge("tools", "tool_calling_llm")

graph_builder = builder.compile()



In [ ]:
#view graph
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage
temp_message = graph_builder.invoke(
    {"messages": [HumanMessage(content="What is addition of 12 and 12?")]}
)

for m in temp_message["messages"]:
    print(m.pretty_print())

In [ ]:
temp_message = graph_builder.invoke(
    {"messages": [HumanMessage(content="and what is their substaction.")]}
)

for m in temp_message["messages"]:
    print(m.pretty_print())

In Above example system did not maintain it's memory from previous question

In below example, we will add memory 

In [7]:
#for adding memory in langgraph
from langgraph.checkpoint.memory import MemorySaver
memory=MemorySaver()

# 2. Build the graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")

# 3. Use the explicit mapping in conditional edges
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
    {
        "tools": "tools",
        "__end__": END
    }
)

# 4. IMPORTANT: Loop back to LLM after tool execution
builder.add_edge("tools", "tool_calling_llm")

#addred memory checkpointer here
graph_builder = builder.compile(checkpointer=memory)



In [8]:
#provided thread id
config={"configurable":{"thread_id":"743434"}}



In [9]:
#using config=config to tell which thread needs to be used
from langchain_core.messages import HumanMessage
temp_message = graph_builder.invoke(
    {"messages": [HumanMessage(content="What is substation of 15 and 12?")]},config=config
)

for m in temp_message["messages"]:
    print(m.pretty_print())

================================ Human Message =================================

What is substation of 15 and 12?
None
================================== Ai Message ==================================
Tool Calls:
  Substract_numbers (call_oz1fYbbZJIZPCIEkXkEsvYbv)
 Call ID: call_oz1fYbbZJIZPCIEkXkEsvYbv
  Args:
    a: 15
    b: 12
None
================================= Tool Message =================================
Name: Substract_numbers

3.0
None
================================== Ai Message ==================================

The result of subtracting 12 from 15 is 3.
None


In [11]:
temp_message = graph_builder.invoke(
    {"messages": [HumanMessage(content="My name is Parth")]},config=config
)

for m in temp_message["messages"]:
    print(m.pretty_print())

================================ Human Message =================================

What is substation of 15 and 12?
None
================================== Ai Message ==================================
Tool Calls:
  Substract_numbers (call_oz1fYbbZJIZPCIEkXkEsvYbv)
 Call ID: call_oz1fYbbZJIZPCIEkXkEsvYbv
  Args:
    a: 15
    b: 12
None
================================= Tool Message =================================
Name: Substract_numbers

3.0
None
================================== Ai Message ==================================

The result of subtracting 12 from 15 is 3.
None
================================ Human Message =================================

what is addition of 150 and 67?
None
================================== Ai Message ==================================
Tool Calls:
  add_numbers (call_jligvNMjczwguUIBS44nBDx2)
 Call ID: call_jligvNMjczwguUIBS44nBDx2
  Args:
    a: 150
    b: 67
None
================================= Tool Message =================================
Name

In [12]:
temp_message = graph_builder.invoke(
    {"messages": [HumanMessage(content=" what is my name?")]},config=config
)

for m in temp_message["messages"]:
    print(m.pretty_print())

================================ Human Message =================================

What is substation of 15 and 12?
None
================================== Ai Message ==================================
Tool Calls:
  Substract_numbers (call_oz1fYbbZJIZPCIEkXkEsvYbv)
 Call ID: call_oz1fYbbZJIZPCIEkXkEsvYbv
  Args:
    a: 15
    b: 12
None
================================= Tool Message =================================
Name: Substract_numbers

3.0
None
================================== Ai Message ==================================

The result of subtracting 12 from 15 is 3.
None
================================ Human Message =================================

what is addition of 150 and 67?
None
================================== Ai Message ==================================
Tool Calls:
  add_numbers (call_jligvNMjczwguUIBS44nBDx2)
 Call ID: call_jligvNMjczwguUIBS44nBDx2
  Args:
    a: 150
    b: 67
None
================================= Tool Message =================================
Name